# 预训练一个迷你GPT （1亿2千4百万参数）

## 基本概念

### GPT 架构

GPT 是一个自回归的语言模型，“自回归”表示它一次吐出一个词元，每个词元都是在前文的基础上生成的。架构基本上就是一系列堆叠的Transformer 解码块。

从输入词元们的IDs到下一个词元的概率，计算流程为：
1. 输入词元ID。形状 （批大小，序列长度）
2. 词元嵌入，将每个词元映射到一个768维的向量。形状（批大小， 序列长度， 768）
3. 位置嵌入，将每个位置同样映射到一个768维的向量。形状与上一步相同。
4. 将词元嵌入和位置嵌入相加。
5. 依次通过12层Transformer块。
6. 归一化。
7. 线性投影回词表，获取各个词的分数。 形状（批大小， 序列长度， 词表大小）
8. Softmax 将分数转换成概率。

### Transformer 块

12个Transformer块模式相同：
1. 层归一化；
2. 多头注意力；
3. 残差连接；
4. 层归一化；
5. 前馈；
6. 残差连接；

### 其他知识点

多头注意力。

KV Cache。

### 推理的两个阶段，预填充和解码

#### 预填充

并行的处理你的完整提示词，所有的词元都已知，所以模型能够对所有的位置同时计算注意力。**这个阶段的瓶颈是算力。** 有大量的矩阵运算。

#### 解码

每次生产一个词元，每个新的词元都需要依赖之前的所有词元。**这个阶段的瓶颈时内存速度。** 模型需要从GPU内存中读取模型权重以及KV 缓存。

### 训练循环

每步训练中：
1. 前向传播。
2. 计算损失。
3. 反向传播。
4. 更新参数。

# 动手构建

## 非Pytorch 版本

In [ ]:
import numpy as np

class Embedding:
    def __init__(self, vocab_size, embed_dim, max_seq_len):
        self.token_embed = np.random.randn(vocab_size, embed_dim) * 0.02
        self.pos_embed = np.random.randn(max_seq_len, embed_dim) * 0.02

    def forward(self, token_ids):
        seq_len = token_ids.shape[-1]
        token_embed = self.token_embed[token_ids]
        pos_emb = self.pos_embed[:seq_len]
        return token_embed + pos_emb

class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.gamma = np.ones(dim)
        self.beta = np.zeros(dim)
        self.eps = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)

        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

class MultiHeadAttention:
    def __init__(self, embed_dim, num_heads):
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.W_q = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_k = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_v = np.random.randn(embed_dim, embed_dim) * 0.02
        self.W_o = np.random.randn(embed_dim, embed_dim) * 0.02

    def forward(self, x, mask=None):
        batch_size, seq_len, embed_dim = x.shape
        Q = (x @ self.W_q).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        K = (x @ self.W_k).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)
        V = (x @ self.W_v).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(0, 2, 1, 3)

        scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(self.head_dim)

        if mask is not None:
            scores = scores + mask

        weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
        weights = weights / weights.sum(axis=-1, keepdims=True)
        attn_out = weights @ V

        attn_out = attn_out.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, embed_dim)
        
        return attn_out @ self.W_o

class FeedForward:
    def __init__(self, embed_dim, ff_dim):
        self.W1 = np.random.randn(embed_dim, ff_dim) * 0.02
        self.b1 = np.zeros(ff_dim)
        self.W2 = np.random.randn(ff_dim, embed_dim) * 0.02
        self.b2 = np.zeros(embed_dim)

    def forward(self, x):
        h = x @ self.W1 + self.b1
        # ReLU 激活函数
        h = np.maximum(0, h)
        return h @ self.W2 + self.b2

class TransformerBlock:
    def __init__(self, embed_dim, num_heads, ff_dim):
        self.ln1 = LayerNorm(embed_dim)
        self.mha = MultiHeadAttention(embed_dim, num_heads)
        self.ln2 = LayerNorm(embed_dim)
        self.ff = FeedForward(embed_dim, ff_dim)

    def forward(self, x, mask=None):
        x = x + self.mha.forward(self.ln1.forward(x), mask)
        x = x + self.ff.forward(self.ln2.forward(x))

        return x

class MiniGPT:
    def __init__(self, vocab_size=50257, embed_dim=768, num_heads=12, num_layers=12, max_seq_len=1024, ff_dim=3072):
        self.embedding = Embedding(vocab_size, embed_dim, max_seq_len)
        self.blocks = [
            TransformerBlock(embed_dim, num_heads, ff_dim)
            for _ in range(num_layers)
        ]
        self.ln_f = LayerNorm(embed_dim)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def forward(self, token_ids):
        seq_len = token_ids.shape[-1]
        mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)

        x = self.embedding.forward(token_ids)
        for block in self.blocks:
            x = block.forward(x, mask)
        x = self.ln_f.forward(x)

        logits = x @ self.embedding.token_embed.T

        return logits

    def count_paramters(self):
        total = 0
        total += self.embedding.token_embed.size
        total += self.embedding.pos_embed.size

        for block in self.blocks:
            total += block.mha.W_q.size + block.mha.W_k.size + block.mha.W_v.size + block.mha.W_o.size
            total += block.ff.W1.size + block.ff.W2.size + block.ff.b1.size + block.ff.b2.size
            total += block.ln1.gamma.size + block.ln1.beta.size
            total += block.ln2.gamma.size + block.ln2.beta.size

        total += self.ln_f.gamma.size + self.ln_f.beta.size

        return total
    

gpt_np = MiniGPT(
    vocab_size=50257,
    embed_dim=768,
    num_heads=12,
    num_layers=12,
    max_seq_len=1024,
    ff_dim=3072
)

print("gpt_np 参数数量：", gpt_np.count_paramters())


def cross_entropy_loss(logits, targets):
    batch_size, seq_len, vocab_size = logits.shape

    logits_flat = logits.reshape(-1, vocab_size)
    targets_flat = targets.reshape(-1)

    max_logits = logits_flat.max(axis=-1, keepdims=True)
    logits_flat = logits_flat - max_logits
    log_softmax = logits_flat - np.log(np.sum(np.exp(logits_flat), axis=-1, keepdims=True))

    loss = -log_softmax[np.arange(batch_size * seq_len), targets_flat].mean()
    return loss


def generate(model, prompt_tokens, max_new_tokens=100, temperature=0.8):
    tokens = list(prompt_tokens)

    seq_len = model.embedding.pos_embed.shape[0]

    for _ in range(max_new_tokens):
        context = np.array(tokens[-seq_len:]).reshape(1, -1)
        logits = model.forward(context)
        next_logits = logits[0, -1, :]

        next_logits = next_logits / temperature
        probs = np.exp(next_logits) / np.sum(np.exp(next_logits))

        next_token = np.random.choice(len(probs), p=probs)
        tokens.append(next_token)

    return tokens


input = [1, 1, 1, 1, 1, 1, 1]
print(input[-5:])

output = generate(gpt_np, input, max_new_tokens=3)

print(output[-5:])
    

gpt_np 参数数量： 124402944
